# Análisis exploratorio de datos: anuncios de vehículos en EE. UU.

Revisión del conjunto `vehicles_us.csv` (anuncios publicados entre mayo de 2018 y abril de 2019) como base para la aplicación web.

In [1]:
import pandas as pd
import plotly.express as px

## 1. Carga de datos

In [2]:
car_data = pd.read_csv('../vehicles_us.csv')
car_data.head()

,price,model_year,model,condition,cylinders,fuel,odometer,transmission,type,paint_color,is_4wd,date_posted,days_listed
0,9400,2011.0,bmw x5,good,6.0,gas,145000.0,automatic,SUV,NaN,1.0,2018-06-23,19
1,25500,NaN,ford f-150,good,6.0,gas,88705.0,automatic,pickup,white,1.0,2018-10-19,50
2,5500,2013.0,hyundai sonata,like new,4.0,gas,110000.0,automatic,sedan,red,NaN,2019-02-07,79
3,1500,2003.0,ford f-150,fair,8.0,gas,NaN,automatic,pickup,NaN,NaN,2019-03-22,9
4,14900,2017.0,chrysler 200,excellent,4.0,gas,80903.0,automatic,sedan,black,NaN,2019-04-02,28


## 2. Estructura general

In [3]:
car_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 51525 entries, 0 to 51524
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   price         51525 non-null  int64  
 1   model_year    47906 non-null  float64
 2   model         51525 non-null  str    
 3   condition     51525 non-null  str    
 4   cylinders     46265 non-null  float64
 5   fuel          51525 non-null  str    
 6   odometer      43633 non-null  float64
 7   transmission  51525 non-null  str    
 8   type          51525 non-null  str    
 9   paint_color   42258 non-null  str    
 10  is_4wd        25572 non-null  float64
 11  date_posted   51525 non-null  str    
 12  days_listed   51525 non-null  int64  
dtypes: float64(4), int64(2), str(7)
memory usage: 7.7 MB


In [4]:
car_data.describe()

,price,model_year,cylinders,odometer,is_4wd,days_listed
count,51525.000000,47906.000000,46265.000000,43633.000000,25572.0,51525.00000
mean,12132.464920,2009.750470,6.125235,115553.461738,1.0,39.55476
std,10040.803015,6.282065,1.660360,65094.611341,0.0,28.20427
min,1.000000,1908.000000,3.000000,0.000000,1.0,0.00000
25%,5000.000000,2006.000000,4.000000,70000.000000,1.0,19.00000
50%,9000.000000,2011.000000,6.000000,113000.000000,1.0,33.00000
75%,16839.000000,2014.000000,8.000000,155000.000000,1.0,53.00000
max,375000.000000,2019.000000,12.000000,990000.000000,1.0,271.00000


## 3. Valores ausentes

In [5]:
missing = car_data.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(car_data) * 100).round(2)
pd.DataFrame({'ausentes': missing, 'porcentaje': missing_pct})

,ausentes,porcentaje
is_4wd,25953,50.37
paint_color,9267,17.99
odometer,7892,15.32
cylinders,5260,10.21
model_year,3619,7.02
condition,0,0.00
model,0,0.00
price,0,0.00
fuel,0,0.00
type,0,0.00


La columna `is_4wd` solo contiene el valor 1 o ausente: el ausente significa que el vehículo **no** es 4x4, así que se rellena con 0. El resto de ausentes (`paint_color`, `odometer`, `cylinders`, `model_year`) se deja tal cual, porque el objetivo es la visualización y no el modelado.

In [6]:
car_data['is_4wd'] = car_data['is_4wd'].fillna(0).astype(int)
car_data['date_posted'] = pd.to_datetime(car_data['date_posted'])
car_data['is_4wd'].value_counts()

is_4wd
0    25953
1    25572
Name: count, dtype: int64

## 4. Duplicados

In [7]:
car_data.duplicated().sum()

np.int64(0)

## 5. Valores atípicos

El precio va de 1 a 375.000 USD y el kilometraje llega a 990.000 millas. Los precios muy bajos corresponden a anuncios con precio de relleno.

In [8]:
fig = px.box(car_data, x='price', title='Precio: diagrama de caja (datos sin filtrar)')
fig.show()

In [9]:
print('Anuncios con precio < 100 USD:', (car_data['price'] < 100).sum())
print('Anuncios con precio > 100.000 USD:', (car_data['price'] > 100000).sum())
print('Anuncios con kilometraje > 500.000:', (car_data['odometer'] > 500000).sum())

Anuncios con precio < 100 USD: 863
Anuncios con precio > 100.000 USD: 17
Anuncios con kilometraje > 500.000: 22


Se excluyen los anuncios con precio fuera del rango 100–100.000 USD y kilometraje superior a 500.000 millas (se conservan los que no tienen kilometraje registrado).

In [10]:
clean_data = car_data[
    car_data['price'].between(100, 100000)
    & ((car_data['odometer'] <= 500000) | car_data['odometer'].isna())
]
excluded = len(car_data) - len(clean_data)
print(f'Registros conservados: {len(clean_data)} | excluidos: {excluded} ({excluded / len(car_data):.2%})')

Registros conservados: 50623 | excluidos: 902 (1.75%)


## 6. Distribución del kilometraje

In [11]:
fig = px.histogram(clean_data, x='odometer', nbins=50, title='Distribución del kilometraje (odometer)')
fig.show()

## 7. Distribución del precio

In [12]:
fig = px.histogram(clean_data, x='price', nbins=50, title='Distribución del precio')
fig.show()

## 8. Kilometraje vs. precio

In [13]:
fig = px.scatter(clean_data, x='odometer', y='price', color='condition', opacity=0.4,
                 title='Kilometraje vs. precio por condición')
fig.show()

In [14]:
clean_data[['odometer', 'price']].corr()

,odometer,price
odometer,1.00000,-0.49469
price,-0.49469,1.00000


## 9. Precio según condición

In [15]:
clean_data.groupby('condition')['price'].median().sort_values()

condition
fair          2500.0
salvage       2500.0
good          7900.0
excellent    10900.0
like new     13995.0
new          22223.0
Name: price, dtype: float64

## 10. Anuncios por tipo de vehículo

In [16]:
type_counts = clean_data['type'].value_counts().reset_index()
type_counts.columns = ['type', 'count']
fig = px.bar(type_counts, x='type', y='count', title='Cantidad de anuncios por tipo de vehículo')
fig.show()

## Conclusiones

- El conjunto tiene 51.525 anuncios y 13 columnas, sin filas duplicadas.
- `is_4wd` tenía la mitad de los valores ausentes, pero en realidad indicaban vehículos sin tracción 4x4; se rellenaron con 0.
- Se excluyó el 1,75 % de los registros por precios de relleno (menos de 100 USD), precios extremos (más de 100.000 USD) o kilometrajes irreales (más de 500.000 millas).
- El kilometraje se concentra alrededor de las 114.000 millas (mediana), y la distribución del precio tiene una cola larga hacia la derecha.
- Hay una relación negativa moderada entre kilometraje y precio (correlación aproximada de −0,49): a más millas, menor precio.
- La condición pesa en el precio: la mediana va de unos 2.500 USD (*fair*, *salvage*) a unos 22.000 USD (*new*).
- SUV, *truck* y *sedan* concentran la mayor parte de los anuncios.
- El histograma de kilometraje y el gráfico de dispersión kilometraje vs. precio pasan a la aplicación web.